In [1]:
import pandas as pd

def get_driver_performance_recent(sessions_list):
    
    if not sessions_list:
        return pd.DataFrame()
    all_race_data = []
    
    for session in sessions_list:
        event_name = session.event['EventName']
        results = session.results.copy()
        
        driver_data = results[["Abbreviation", "Position", "Points"]].copy()
        driver_data.rename(columns={"Abbreviation": "Driver"}, inplace=True)
        driver_data["Event"] = event_name
        
        all_race_data.append(driver_data)
        
    combined_df = pd.concat(all_race_data, ignore_index=True)
    
    performance_summary = combined_df.pivot(
        index="Driver", 
        columns="Event", 
        values=["Position", "Points"]
    )
    
    performance_summary.columns = [f"{col[0]}_{col[1]}" for col in performance_summary.columns]
    
    return performance_summary.reset_index()


def get_constructor_performance_recent(sessions_list):

    if not sessions_list:
        return pd.DataFrame()
        
    all_constructor_data = []
    
    for session in sessions_list:
        event_name = session.event['EventName']
        results = session.results.copy()
        
        team_points = results.groupby("TeamName")["Points"].sum().reset_index()
        team_points.rename(columns={"TeamName": "Constructor"}, inplace=True)
        team_points["Event"] = event_name
        
        all_constructor_data.append(team_points)
        
    combined_df = pd.concat(all_constructor_data, ignore_index=True)
    
    constructor_summary = combined_df.pivot(
        index="Constructor", 
        columns="Event", 
        values="Points"
    )
    
    if isinstance(constructor_summary.columns, pd.MultiIndex):
        constructor_summary.columns = [f"Points_{col[1]}" for col in constructor_summary.columns]
    else:
        constructor_summary.columns = [f"Points_{col}" for col in constructor_summary.columns]
    
    return constructor_summary.reset_index()

In [41]:
Map = {
    "NOR":"McLaren",
    "PIA":"McLaren",

    "VER":"Red Bull Racing",
    "HAD":"Red Bull Racing",

    "LEC":"Ferrari",
    "HAM":"Ferrari",

    "RUS":"Mercedes",
    "ANT":"Mercedes",

    "ALO":"Aston Martin",
    "STR":"Aston Martin",

    "ALB":"Williams",
    "SAI":"Williams",

    "OCO":"Haas F1 Team",
    "BEA":"Haas F1 Team",

    "GAS":"Alpine",
    "COL":"Alpine",

    "HUL":"Audi",
    "BOR":"Audi",

    "PER":"Cadillac",
    "BOT":"Cadillac",
    
    "LAW":"Racing Bulls",
    "LIN":"Racing Bulls"  
}

In [16]:
import fastf1

season = 2026
race = 4 

sessions_to_analyze = []

start_round = max(1, race - 2)

for r in range(start_round, race + 1):
    Result = fastf1.get_session(season, r, 'R')
    Result.load()
    sessions_to_analyze.append(Result)
    
driver_form = get_driver_performance_recent(sessions_to_analyze)
constructor_form = get_constructor_performance_recent(sessions_to_analyze)

driver_form.iloc[:, 4:7] = driver_form.iloc[:, 4:7].fillna(0)
driver_form.iloc[:,1:4] = driver_form.iloc[:,1:4].fillna(22)
driver_form['AveragePositionFromLast3Races'] = driver_form.iloc[:, 1:4].mean(axis=1)
driver_form['AveragePointsFromLast3Races'] = driver_form.iloc[:, 4:7].mean(axis=1)


constructor_form["ConstructorAveragePointFromLast3Races"] = constructor_form.iloc[:,1:4].mean(axis=1)

driver_form['Constructor'] = driver_form['Driver'].map(Map)

driver_form = driver_form.merge(
    constructor_form[['Constructor','ConstructorAveragePointFromLast3Races']],
    on = 'Constructor',
    how = 'left'
)

final_driver_form = driver_form[['Driver','Constructor','AveragePositionFromLast3Races','AveragePointsFromLast3Races','ConstructorAveragePointFromLast3Races']]

final_driver_form


core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 12 completed the race distance 00:00.022000 before the recorded end of the session.
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '87', '10', '30', '6', '55', '43', '27', '41', '77', '31', '11', '3', '14', '18', '81', '1', '5', '23']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for 

,Driver,Constructor,AveragePositionFromLast3Races,AveragePointsFromLast3Races,ConstructorAveragePointFromLast3Races
0,ALB,Williams,17.333333,0.333333,1.666667
1,ALO,Aston Martin,16.666667,0.000000,0.000000
2,ANT,Mercedes,1.000000,25.000000,39.000000
3,BEA,Haas F1 Team,12.666667,3.333333,3.666667
4,BOR,Audi,15.333333,0.000000,0.000000
5,BOT,Cadillac,16.666667,0.000000,0.000000
6,COL,Alpine,11.000000,2.333333,7.000000
7,GAS,Alpine,11.333333,4.666667,7.000000
8,HAD,Red Bull Racing,14.000000,1.333333,6.000000
9,HAM,Ferrari,5.000000,10.333333,20.666667


In [3]:
driver_form.iloc[:, 4:7] = driver_form.iloc[:, 4:7].fillna(0)

In [4]:
driver_form.iloc[:,1:4] = driver_form.iloc[:,1:4].fillna(22)

In [5]:
driver_form['AveragePositionFromLast3Races'] = driver_form.iloc[:, 1:4].mean(axis=1)

In [6]:
driver_form['AveragePointsFromLast3Races'] = driver_form.iloc[:, 4:7].mean(axis=1)

In [7]:
driver_form

,Driver,Position_Chinese Grand Prix,Position_Japanese Grand Prix,Position_Miami Grand Prix,Points_Chinese Grand Prix,Points_Japanese Grand Prix,Points_Miami Grand Prix,AveragePositionFromLast3Races,AveragePointsFromLast3Races
0,ALB,22.0,20.0,10.0,0.0,0.0,1.0,17.333333,0.333333
1,ALO,17.0,18.0,15.0,0.0,0.0,0.0,16.666667,0.000000
2,ANT,1.0,1.0,1.0,25.0,25.0,25.0,1.000000,25.000000
3,BEA,5.0,22.0,11.0,10.0,0.0,0.0,12.666667,3.333333
4,BOR,21.0,13.0,12.0,0.0,0.0,0.0,15.333333,0.000000
5,BOT,13.0,19.0,18.0,0.0,0.0,0.0,16.666667,0.000000
6,COL,10.0,16.0,7.0,1.0,0.0,6.0,11.000000,2.333333
7,GAS,6.0,7.0,21.0,8.0,6.0,0.0,11.333333,4.666667
8,HAD,8.0,12.0,22.0,4.0,0.0,0.0,14.000000,1.333333
9,HAM,3.0,6.0,6.0,15.0,8.0,8.0,5.000000,10.333333


In [8]:
constructor_form["ConstructorAveragePointFromLast3Races"] = constructor_form.iloc[:,1:4].mean(axis=1)

In [9]:
constructor_form

,Constructor,Points_Chinese Grand Prix,Points_Japanese Grand Prix,Points_Miami Grand Prix,ConstructorAveragePointFromLast3Races
0,Alpine,9.0,6.0,6.0,7.000000
1,Aston Martin,0.0,0.0,0.0,0.000000
2,Audi,0.0,0.0,0.0,0.000000
3,Cadillac,0.0,0.0,0.0,0.000000
4,Ferrari,27.0,23.0,12.0,20.666667
5,Haas F1 Team,10.0,1.0,0.0,3.666667
6,McLaren,0.0,28.0,33.0,20.333333
7,Mercedes,43.0,37.0,37.0,39.000000
8,Racing Bulls,6.0,2.0,0.0,2.666667
9,Red Bull Racing,4.0,4.0,10.0,6.000000


In [10]:
Map = {
    "NOR":"McLaren",
    "PIA":"McLaren",

    "VER":"Red Bull Racing",
    "HAD":"Red Bull Racing",

    "LEC":"Ferrari",
    "HAM":"Ferrari",

    "RUS":"Mercedes",
    "ANT":"Mercedes",

    "ALO":"Aston Martin",
    "STR":"Aston Martin",

    "ALB":"Williams",
    "SAI":"Williams",

    "OCO":"Haas F1 Team",
    "BEA":"Haas F1 Team",

    "GAS":"Alpine",
    "COL":"Alpine",

    "HUL":"Audi",
    "BOR":"Audi",

    "PER":"Cadillac",
    "BOT":"Cadillac",
    
    "LAW":"Racing Bulls",
    "LIN":"Racing Bulls"  
}

In [14]:
driver_form['Constructor'] = driver_form['Driver'].map(Map)

driver_form = driver_form.merge(
    constructor_form[['Constructor','ConstructorAveragePointFromLast3Races']],
    on = 'Constructor',
    how = 'left'
)

final_driver_form = driver_form[['Driver','Constructor','AveragePositionFromLast3Races','AveragePointsFromLast3Races','ConstructorAveragePointFromLast3Races']]

In [15]:
final_driver_form

,Driver,Constructor,AveragePositionFromLast3Races,AveragePointsFromLast3Races,ConstructorAveragePointFromLast3Races
0,ALB,Williams,17.333333,0.333333,1.666667
1,ALO,Aston Martin,16.666667,0.000000,0.000000
2,ANT,Mercedes,1.000000,25.000000,39.000000
3,BEA,Haas F1 Team,12.666667,3.333333,3.666667
4,BOR,Audi,15.333333,0.000000,0.000000
5,BOT,Cadillac,16.666667,0.000000,0.000000
6,COL,Alpine,11.000000,2.333333,7.000000
7,GAS,Alpine,11.333333,4.666667,7.000000
8,HAD,Red Bull Racing,14.000000,1.333333,6.000000
9,HAM,Ferrari,5.000000,10.333333,20.666667


In [ ]:
def get_driver_points(Result):

    driver_points = Result.results[["Abbreviation", "Points"]].copy()
    
    driver_points.rename(columns={"Abbreviation": "Driver"}, inplace=True)
    
    driver_points = driver_points.sort_values(by="Points", ascending=False).reset_index(drop=True)
    
    return driver_points

def get_constructor_points(Result):

    results = Result.results.copy()
    
    constructor_points = results.groupby("TeamName")["Points"].sum().reset_index()
    
    constructor_points.rename(columns={"TeamName": "Constructor"}, inplace=True)
    
    constructor_points = constructor_points.sort_values(by="Points", ascending=False).reset_index(drop=True)
    
    return constructor_points

In [31]:
season = 2026
race = 4
Result = fastf1.get_session(season, race, 'R')
Result.load()

core           INFO 	Loading data for Miami Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 12 completed the race distance 00:00.027000 before the recorded end of the session.
core           INFO 	Finished loading data for 22 drivers: ['12', '1', '81', '63', '3', '44', '43', '16', '55', '23', '87', '5',

In [37]:
Driver_points = get_driver_points(Result)
Constructor_points = get_constructor_points(Result)

In [38]:
Driver_points

,Driver,Points
0,ANT,25.0
1,NOR,18.0
2,PIA,15.0
3,RUS,12.0
4,VER,10.0
5,HAM,8.0
6,COL,6.0
7,LEC,4.0
8,SAI,2.0
9,ALB,1.0


In [39]:
Constructor_points

,Constructor,Points
0,Mercedes,37.0
1,McLaren,33.0
2,Ferrari,12.0
3,Red Bull Racing,10.0
4,Alpine,6.0
5,Williams,3.0
6,Aston Martin,0.0
7,Audi,0.0
8,Cadillac,0.0
9,Haas F1 Team,0.0


In [40]:
Driver_points['Constructor'] = Driver_points['Driver'].map(Map)

Driver_points = Driver_points.merge(
    Constructor_points[['Constructor','Points']],
    on = 'Constructor',
    how = 'left'
)
Driver_points.rename(columns={'Points_x':'DriverPoints','Points_y':'ConstructorPoints'},inplace=True)
Driver_points

,Driver,DriverPoints,Constructor,ConstructorPoints
0,ANT,25.0,Mercedes,37.0
1,NOR,18.0,McLaren,33.0
2,PIA,15.0,McLaren,33.0
3,RUS,12.0,Mercedes,37.0
4,VER,10.0,Red Bull Racing,10.0
5,HAM,8.0,Ferrari,12.0
6,COL,6.0,Alpine,6.0
7,LEC,4.0,Ferrari,12.0
8,SAI,2.0,Williams,3.0
9,ALB,1.0,Williams,3.0


In [47]:
import fastf1
from fastf1.ergast import Ergast
import pandas as pd

def get_driver_points_before_race(season, race_name):
    # 1. Look up the event to find out which Round Number it is
    event = fastf1.get_event(season, race_name)
    current_round = event.RoundNumber
    
    ergast = Ergast()
    
    # 2. If it's the first race of the season, fetch the driver list and set points to 0
    if current_round == 1:
        # In newer FastF1 versions, this returns an ErgastSimpleResponse 
        # which acts directly as a Pandas DataFrame! No .content[0] needed.
        drivers = ergast.get_driver_info(season=season, round=1)
        
        driver_points = drivers[["driverCode"]].copy()
        driver_points.rename(columns={"driverCode": "Driver"}, inplace=True)
        driver_points["Points"] = 0.0
        
        return driver_points.sort_values(by="Driver").reset_index(drop=True)
        
    # 3. For round 2 onwards, fetch the championship standings from the PREVIOUS round
    standings_response = ergast.get_driver_standings(season=season, round=current_round - 1)
    
    # 4. Safely handle both ErgastMultiResponse and ErgastSimpleResponse structures
    if hasattr(standings_response, 'content'):
        standings = standings_response.content[0]
    else:
        standings = standings_response
    
    # 5. Clean up the dataframe to match your original format
    driver_points = standings[["driverCode", "points"]].copy()
    driver_points.rename(columns={"driverCode": "Driver", "points": "Points"}, inplace=True)
    
    # 6. Sort the standings by points (highest to lowest)
    driver_points = driver_points.sort_values(by="Points", ascending=False).reset_index(drop=True)
    
    return driver_points

# --- Example Usage ---
df_points = get_driver_points_before_race(2026, "Canada")
# print(df_points.head())

In [61]:
df_points['Constructor'] = df_points['Driver'].map(Map)
df_points

,Driver,Points,Constructor
0,ANT,100.0,Mercedes
1,RUS,80.0,Mercedes
2,LEC,59.0,Ferrari
3,NOR,51.0,McLaren
4,HAM,51.0,Ferrari
5,PIA,43.0,McLaren
6,VER,26.0,Red Bull Racing
7,BEA,17.0,Haas F1 Team
8,GAS,16.0,Alpine
9,LAW,10.0,Racing Bulls


In [57]:
import fastf1
from fastf1.ergast import Ergast
import pandas as pd

def get_constructor_points_before_race(season, race_name):
    # 1. Look up the event to find out which Round Number it is
    event = fastf1.get_event(season, race_name)
    current_round = event.RoundNumber
    
    ergast = Ergast()
    
    # 2. If it's the first race of the season, fetch the constructor list and set points to 0
    if current_round == 1:
        # Fetch the constructors registered for round 1
        constructors = ergast.get_constructor_info(season=season, round=1)
        
        # We use 'constructorId' (e.g., 'ferrari', 'red_bull') as the identifier
        constructor_points = constructors[["constructorId"]].copy()
        constructor_points.rename(columns={"constructorId": "Constructor"}, inplace=True)
        constructor_points["Points"] = 0.0
        
        return constructor_points.sort_values(by="Constructor").reset_index(drop=True)
        
    # 3. For round 2 onwards, fetch the constructor standings from the PREVIOUS round
    standings_response = ergast.get_constructor_standings(season=season, round=current_round - 1)
    
    # 4. Safely handle both ErgastMultiResponse and ErgastSimpleResponse structures
    if hasattr(standings_response, 'content'):
        standings = standings_response.content[0]
    else:
        standings = standings_response
    
    # 5. Clean up the dataframe
    constructor_points = standings[["constructorId", "points"]].copy()
    constructor_points.rename(columns={"constructorId": "Constructor", "points": "Points"}, inplace=True)
    
    # 6. Sort the standings by points (highest to lowest)
    constructor_points = constructor_points.sort_values(by="Points", ascending=False).reset_index(drop=True)
    
    return constructor_points

# --- Example Usage ---
df_constructor_points = get_constructor_points_before_race(2026, "Canada")
# print(df_constructor_points.head())

In [58]:
df_constructor_points

,Constructor,Points
0,mercedes,180.0
1,ferrari,110.0
2,mclaren,94.0
3,red_bull,30.0
4,alpine,23.0
5,haas,18.0
6,rb,14.0
7,williams,5.0
8,audi,2.0
9,cadillac,0.0


In [59]:
Constructor_Map = {
    "mercedes":"Mercedes",
    "ferrari":"Ferrari",
    "mclaren":"McLaren",
    "red_bull":"Red Bull Racing",
    "alpine":"Alpine",
    "haas":"Haas F1 Team",
    "rb":"Racing Bulls",
    "williams":"Williams",
    "audi":"Audi",
    "cadillac":"Cadillac",
    "aston_martin":"Aston Martin"
}

In [60]:
df_constructor_points['NewConstructor'] = df_constructor_points['Constructor'].map(Constructor_Map)
df_constructor_points.drop(columns=['Constructor'],axis=1,inplace=True)
df_constructor_points.rename({'NewConstructor':'Constructor'},axis=1,inplace=True)
df_constructor_points

,Points,Constructor
0,180.0,Mercedes
1,110.0,Ferrari
2,94.0,McLaren
3,30.0,Red Bull Racing
4,23.0,Alpine
5,18.0,Haas F1 Team
6,14.0,Racing Bulls
7,5.0,Williams
8,2.0,Audi
9,0.0,Cadillac


In [62]:
df_points = df_points.merge(
    df_constructor_points[["Constructor","Points"]],
    on = "Constructor",
    how = "left"
)

In [65]:
df_points.rename({"Points_x":"DriversPoint","Points_y":"ConstructorsPoint"},axis=1,inplace=True)
df_points

,Driver,DriversPoint,Constructor,ConstructorsPoint
0,ANT,100.0,Mercedes,180.0
1,RUS,80.0,Mercedes,180.0
2,LEC,59.0,Ferrari,110.0
3,NOR,51.0,McLaren,94.0
4,HAM,51.0,Ferrari,110.0
5,PIA,43.0,McLaren,94.0
6,VER,26.0,Red Bull Racing,30.0
7,BEA,17.0,Haas F1 Team,18.0
8,GAS,16.0,Alpine,23.0
9,LAW,10.0,Racing Bulls,14.0
